# AnimationStudio - Phase 7: Production Planning & Storyboard System

Transforms a generated story into a **production-ready blueprint** that
answers the question: *"exactly what needs to be generated?"*

Every episode is decomposed into a structured production plan - scene and
shot schema, camera planning, prompt per shot, render queue, and continuity
validation - **before any GPU/ComfyUI work begins**. This is the studio's
pre-production department.

This notebook runs the **entire Phase 7 chain with in-process mock backends**
(no GPU, no ComfyUI, no network): story generation -> blueprint -> episode ->
prompts -> render queue -> the 8-step episode workflow drive to COMPLETED.
It then writes `PHASE7_REPORT.md` and can run the Phase-7 test suites.

## Steps

1. Runtime -> Change runtime type -> T4 GPU (or better) - recommended, but
   this notebook is offline-safe: every cell runs with mocks.
2. In Cell 1 set `REPO_URL` to your GitHub clone URL.
3. Runtime -> Run all.


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (colab-gpu is the only supported branch).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}

# colab-gpu -> fp8 Flux (16GB VRAM, best on T4).  master is deprecated/unused.
BRANCH = "colab-gpu"  #@param ["colab-gpu"]

# Story generation scope.  The default seed produces a small deterministic
# episode; raise EPISODES for a multi-episode batch plan.
SEASON = 1  #@param {type:"integer"}
EPISODE_NUMBER = 1  #@param {type:"integer"}
EPISODES = 1  #@param {type:"integer"}

# Where to write the production report (relative to the repo checkout).
REPORT_PATH = "PHASE7_REPORT.md"  #@param {type:"string"}

# Optional: catalog.db on Google Drive so character enrichment matches the
# locked universe.  Off = local catalog.db (empty/universe-free) still works
# because every production step below is mock-driven.
DRIVE_ROOT = "/content/drive/MyDrive/AnimationStudio"  #@param {type:"string"}
USE_DRIVE_DB = True  #@param {type:"boolean"}
DB = f"{DRIVE_ROOT}/catalog.db" if USE_DRIVE_DB else "catalog.db"

# Run the Phase-7 test suites after building the report (Cell 7).
RUN_TESTS = True  #@param {type:"boolean"}

# Cell 8: push the refreshed report back to GitHub. Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 1 to your GitHub repository before running.")


In [ ]:
#@title 2. Mount Google Drive (catalog.db only)

if USE_DRIVE_DB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print("Drive ready (holds catalog.db only):", DRIVE_ROOT)
else:
    print("Drive not used - running against local catalog.db (mock-ready).")


In [ ]:
#@title 3. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

# Studio first (torch is already preinstalled on Colab), then light deps.
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn", "timm", "diffusers", "transformers"])
print("Studio installed (branch:", BRANCH, ")")


In [ ]:
#@title 4. Preview the production scope (no generation)

# Reads the checked-out Universe + character catalog so the story engine
# enriches the episode with real characters/locations.  Offline: uses what
# is in the checkout, never the network.

import sys
sys.path.insert(0, REPO)

from src.universe.catalog import discover_characters

try:
    seeds = discover_characters(f"{REPO}/Universe")
    chars = [s.name for s in seeds]
except Exception as exc:
    chars = []
    print(f"  (could not enumerate Universe: {exc})")

print("=" * 72)
print("  PRODUCTION PLANNING PREVIEW")
print("=" * 72)
print(f"  Seasons:         {SEASON}..{SEASON + EPISODES - 1} (episodes {EPISODE_NUMBER}..{EPISODE_NUMBER})")
print(f"  Characters:      {len(chars)} in Universe")
print(f"  Backend:         mock (in-process) - no GPU/ComfyUI required")
print(f"  Report:          {REPORT_PATH}")
print("=" * 72)
print("  Each episode becomes:")
print("    - blueprint  -> episode model (scenes/shots/camera)")
print("    - per-shot prompts via ProductionPipeline.generate_prompts")
print("    - render queue via ProductionPipeline.build_render_queue")
print("    - 8-step episode workflow driven to COMPLETED (mock)")
print()
print("  This is the pre-production pass.  Real image generation happens in")
print("  AnimationStudio_Colab_Phase8.ipynb (or the Phase-1/2/3 notebooks).")


In [ ]:
#@title 5. Story -> production blueprint (mock) 

# Phase 7 core: decompose a generated story into a structured production
# plan.  All drivers are in-process; no GPU or network involved.

import sys
sys.path.insert(0, REPO)

from src.story_engine.generator import EpisodeGenerator
from src.production.blueprint_adapter import blueprint_to_episode

gen = EpisodeGenerator(catalog_path=DB)

blueprints = []
episodes = []
for n in range(EPISODES):
    bp = gen.generate_episode(
        season=SEASON,
        episode_number=EPISODE_NUMBER + n,
    )
    blueprints.append(bp)
    ep = blueprint_to_episode(bp)
    episodes.append(ep)
    print(f"  episode {ep.id}: '{bp.title}' - {ep.scene_count} scenes, "
          f"{ep.shot_count} shots, {ep.duration_seconds}s")

# Validation is the quality gate: an incomplete blueprint raises.
for bp in blueprints:
    issues = bp.validate()
    if issues:
        print(f"  ! blueprint {bp.episode_id} validation:", "; ".join(issues))
print("Production blueprints ready:", len(episodes))


In [ ]:
#@title 6. Per-shot prompts, render queue, and continuity QC

# Turns each episode into the exact list of generation tasks: one prompt per
# shot + a render queue + continuity validation (Phase 7 deliverables).

from src.production.pipeline import ProductionPipeline

pipeline = ProductionPipeline()

print("=" * 72)
print("  SHOT / SCENE / PROMPT PLAN")
print("=" * 72)

for ep in episodes:
    prompts = pipeline.generate_prompts(ep)
    queue = pipeline.build_render_queue(ep)
    issues = pipeline.validate_continuity(ep)

    print(f"\n### Episode {ep.id} - '{ep.title}' ({len(prompts)} shots)")
    for shot_id, prompt in list(prompts.items())[:6]:
        print(f"  - {shot_id}: {prompt[:100]}...")
    if len(prompts) > 6:
        print(f"  ... and {len(prompts) - 6} more")
    print(f"  render queue: {len(queue)} tasks "
          f"({sum(1 for t in queue if t.status == 'queued')} queued)")
    print(f"  continuity issues: {len(issues)}")
    for iss in issues[:4]:
        print(f"    - {iss}")
print()
print("Phase 7 planning complete - ready for image generation (Phase 8).")


In [ ]:
#@title 7. Drive the 8-step episode workflow + write PHASE7_REPORT.md

# Orchestrates the EpisodeWorkflowFactory 8-step pipeline to COMPLETED with
# mock backends (proves the wiring end-to-end), then writes the Phase 7
# production report into the checkout.

from src.studio.orchestrator import PipelineOrchestrator
from src.studio.workflow import EpisodeWorkflowFactory

factory = EpisodeWorkflowFactory()
orch = PipelineOrchestrator()
orch.setup_defaults()

print("Episode workflow steps:", [s[0] for s in factory.STEP_SEQUENCE])

results = []
total_issues = 0
for ep in episodes:
    orch.create_pipeline(ep.id)
    complete = orch.process_pipeline(ep.id, passes=20)
    done = orch.pipeline(ep.id)
    results.append((ep, complete, done))
    total_issues += len(pipeline.validate_continuity(ep))
    print(f"  {ep.id}: COMPLETED={complete} completed_steps="
          f"{sum(1 for s in done.steps if s.status.name == 'COMPLETED')}/{len(done.steps)}")

# ---------------------------------------------------------------- report ---
report_lines = [
    "# PHASE7_REPORT.md",
    "",
    "## Summary",
    "",
    f"- Episodes planned: {len(episodes)}",
    f"- Total shots: {sum(ep.shot_count for ep in episodes)}",
    f"- Backend: mock (in-process) — real generation in Phase 8",
    "",
    "## Episode Production Plans",
    "",
]
for ep, complete, done in results:
    report_lines += [
        f"### {ep.id} — {ep.title}",
        "",
        f"- Scenes: {ep.scene_count} | Shots: {ep.shot_count} | "
        f"Duration: {ep.duration_seconds}s",
        f"- Workflow completed: {complete}",
        f"- Manifest: age {ep.manifest.target_age}, "
        f"'{ep.manifest.learning_goal}', "
        f"has_song={ep.manifest.has_song}",
        "",
        "| Shot | Camera | Movement | Prompt |",
        "| --- | --- | --- | --- |",
    ]
    prompts = pipeline.generate_prompts(ep)
    for scene in ep.scenes[:8]:
        for shot in scene.shots:
            shot_id = shot.id
            prompt = prompts.get(shot_id, "")
            camera = shot.camera
            report_lines.append(
                f"| {shot_id} | {camera.shot_type} | {camera.movement} | "
                f"{prompt[:90]} |"
            )
    report_lines += [""]

report_lines += [
    "## Validation",
    "",
    f"- {total_issues} continuity issue(s) across all planned episodes.",
    "",
]

import datetime
report_lines += [
    "---",
    f"Generated by AnimationStudio_Colab_Phase7.ipynb at {datetime.datetime.now():%Y-%m-%d %H:%M}.",
]

with open(f"{REPO}/{REPORT_PATH}", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print("\nWrote", REPORT_PATH, f"({len(report_lines)} lines)")


In [ ]:
#@title 8. Run the Phase-7 test suites

if RUN_TESTS:
    os.chdir(REPO)
    !python -m pytest tests/test_story_to_production_integration.py tests/test_production.py tests/test_studio.py -q --timeout=60
else:
    print("RUN_TESTS is off - skipping Phase-7 suites.")


In [ ]:
#@title 9. Sync the refreshed report (GitHub push or manual download)

from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import auto_sync

    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL,
              message=f"Phase 7 production plan {datetime.now():%Y-%m-%d %H:%M}")
else:
    import zipfile
    from google.colab import files

    zip_path = f"{WORK}/phase7_report.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{REPO}/{REPORT_PATH}", REPORT_PATH)
    files.download(zip_path)
    print("Downloaded phase7_report.zip (PHASE7_REPORT.md).")


## Next steps

- **Open `PHASE7_REPORT.md`** (pushed to GitHub by Cell 9) to review the
  scene/shot/prompt plan before spending GPU.
- **Run Phase 8** (`AnimationStudio_Colab_Phase8.ipynb`) to turn these
  per-shot prompts into actual images.
- **Run Phases 1-3** first if the character/world libraries are not yet
  generated - Phase 8 image prompts work best against locked references.
- Adjust `SEASON` / `EPISODE_NUMBER` / `EPISODES` in Cell 1 and re-run to
  re-plan a different episode.

## Troubleshooting

- `catalog.db` not found on Drive: set `USE_DRIVE_DB = False` in Cell 1 (all
  production steps still run with mocks).
- Continuity issues in the report: expected at this stage - they are the
  input to Phase 8 refinement, not a failure.
- Scripts are missing from the checkout: re-run Cell 3 (`git pull`).
